# 02 - Enriquecimiento y unificacion

Objetivos de este notebook:
- Integrar Taxi Zones para origen (PU) y destino (DO).
- Unificar viajes de Yellow y Green en una estructura comun.
- Normalizar catalogos operativos (vendor, rate code, payment type, trip type, store and forward).
- Publicar tablas curadas en Snowflake.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os

In [ ]:
app = (
    SparkSession.builder
    .appName('02_enriquecimiento_y_unificacion')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)

# Credenciales Snowflake desde variables de ambiente
sf_option = {
    'sfURL': os.environ['SF_HOST'],
    'sfUser': os.environ['SF_USER'],
    'sfPassword': os.environ['SF_PASSWORD'],
    'sfDatabase': os.environ['SF_DATABASE'],
    'sfSchema': os.environ.get('SF_RAW_SCHEMA', 'raw'),
    'sfWarehouse': os.environ.get('SF_WAREHOUSE', ''),
    'sfRole': os.environ.get('SF_ROLE', ''),
}

In [ ]:
# Lectura de tablas RAW generadas en el notebook 01
df_yellow_raw = (
    app.read.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'RAW.YELLOW_TRIPS')
    .load()
)

df_green_raw = (
    app.read.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'RAW.GREEN_TRIPS')
    .load()
)

print('Yellow rows:', df_yellow_raw.count())
print('Green rows:', df_green_raw.count())

In [ ]:
# Estandarizacion de timestamps y claves operativas para unificar tipos de taxi
def base_columns(df):
    pickup_col = F.coalesce(
        F.col('tpep_pickup_datetime'),
        F.col('lpep_pickup_datetime'),
        F.col('pickup_datetime')
    )
    dropoff_col = F.coalesce(
        F.col('tpep_dropoff_datetime'),
        F.col('lpep_dropoff_datetime'),
        F.col('dropoff_datetime')
    )

    vendor_col = F.coalesce(F.col('VendorID'), F.col('vendorid')).cast('int')
    rate_col = F.coalesce(F.col('RatecodeID'), F.col('rate_code')).cast('int')
    payment_col = F.col('payment_type').cast('int')

    # trip_type only exists in Green taxi data; use lit(None) if column is missing
    if 'trip_type' in df.columns:
        trip_type_col = F.col('trip_type').cast('int')
    else:
        trip_type_col = F.lit(None).cast('int')

    return (
        df
        .withColumn('pickup_datetime_utc', pickup_col.cast('timestamp'))
        .withColumn('dropoff_datetime_utc', dropoff_col.cast('timestamp'))
        .withColumn('vendor_id', vendor_col)
        .withColumn('rate_code_id', rate_col)
        .withColumn('payment_type_code', payment_col)
        .withColumn('trip_type_code', trip_type_col)
        .withColumn('store_and_fwd_flag_norm', F.upper(F.trim(F.col('store_and_fwd_flag'))))
    )

In [ ]:
# Se aplica el mismo esquema de negocio para yellow y green
df_yellow = base_columns(df_yellow_raw).withColumn('taxi_type', F.lit('yellow'))
df_green = base_columns(df_green_raw).withColumn('taxi_type', F.lit('green'))

unified_cols = [
    'trip_nk', 'run_id', 'service_type', 'taxi_type',
    'source_year', 'source_month', 'ingested_at_utc', 'source_path',
    'pickup_datetime_utc', 'dropoff_datetime_utc',
    'PULocationID', 'DOLocationID',
    'vendor_id', 'rate_code_id', 'payment_type_code', 'trip_type_code', 'store_and_fwd_flag_norm',
    'passenger_count', 'trip_distance',
    'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
    'improvement_surcharge', 'congestion_surcharge', 'airport_fee', 'total_amount'
]

for c in unified_cols:
    if c not in df_yellow.columns:
        df_yellow = df_yellow.withColumn(c, F.lit(None))
    if c not in df_green.columns:
        df_green = df_green.withColumn(c, F.lit(None))

df_trips_unified = df_yellow.select(*unified_cols).unionByName(df_green.select(*unified_cols))

In [ ]:
# Integracion Taxi Zones desde lookup oficial TLC
zones_url = os.environ.get('TAXI_ZONE_PATH', 'https://s3.amazonaws.com/nyc-tlc/misc/taxi+_zone_lookup.csv')

df_zones = (
    app.read.option('header', True).option('inferSchema', True).csv(zones_url)
    .withColumn('LocationID', F.col('LocationID').cast('int'))
    .withColumn('Borough', F.trim(F.col('Borough')))
    .withColumn('Zone', F.trim(F.col('Zone')))
    .withColumn('service_zone', F.trim(F.col('service_zone')))
    .dropDuplicates(['LocationID'])
)

df_zones_pu = (
    df_zones
    .select(
        F.col('LocationID').alias('pu_location_id'),
        F.col('Borough').alias('pu_borough'),
        F.col('Zone').alias('pu_zone'),
        F.col('service_zone').alias('pu_service_zone')
    )
)

df_zones_do = (
    df_zones
    .select(
        F.col('LocationID').alias('do_location_id'),
        F.col('Borough').alias('do_borough'),
        F.col('Zone').alias('do_zone'),
        F.col('service_zone').alias('do_service_zone')
    )
)

In [ ]:
# Normalizacion de catalogos (codigos a descripciones)
catalog_vendor = [
    (1, 'Creative Mobile Technologies'),
    (2, 'VeriFone Inc.'),
    (6, 'Myle Technologies Inc.')
]

catalog_ratecode = [
    (1, 'Standard rate'),
    (2, 'JFK'),
    (3, 'Newark'),
    (4, 'Nassau or Westchester'),
    (5, 'Negotiated fare'),
    (6, 'Group ride')
]

catalog_payment_type = [
    (1, 'Credit card'),
    (2, 'Cash'),
    (3, 'No charge'),
    (4, 'Dispute'),
    (5, 'Unknown'),
    (6, 'Voided trip')
]

catalog_trip_type = [
    (1, 'Street-hail'),
    (2, 'Dispatch')
]

catalog_store_and_fwd = [
    ('Y', 'Stored and forwarded'),
    ('N', 'Not a stored trip')
]

df_cat_vendor = app.createDataFrame(catalog_vendor, ['vendor_id', 'vendor_desc'])
df_cat_rate = app.createDataFrame(catalog_ratecode, ['rate_code_id', 'rate_code_desc'])
df_cat_payment = app.createDataFrame(catalog_payment_type, ['payment_type_code', 'payment_type_desc'])
df_cat_trip = app.createDataFrame(catalog_trip_type, ['trip_type_code', 'trip_type_desc'])
df_cat_saf = app.createDataFrame(catalog_store_and_fwd, ['store_and_fwd_flag_norm', 'store_and_fwd_desc'])

In [ ]:
df_trips_enriched = (
    df_trips_unified
    .withColumn('PULocationID', F.col('PULocationID').cast('int'))
    .withColumn('DOLocationID', F.col('DOLocationID').cast('int'))
    .join(df_zones_pu, F.col('PULocationID') == F.col('pu_location_id'), 'left')
    .join(df_zones_do, F.col('DOLocationID') == F.col('do_location_id'), 'left')
    .join(df_cat_vendor, on='vendor_id', how='left')
    .join(df_cat_rate, on='rate_code_id', how='left')
    .join(df_cat_payment, on='payment_type_code', how='left')
    .join(df_cat_trip, on='trip_type_code', how='left')
    .join(df_cat_saf, on='store_and_fwd_flag_norm', how='left')
    .withColumn('trip_duration_minutes', (F.unix_timestamp('dropoff_datetime_utc') - F.unix_timestamp('pickup_datetime_utc')) / 60.0)
    .withColumn('trip_date', F.to_date('pickup_datetime_utc'))
)

In [ ]:
# Publicacion de tablas curadas
(
    df_zones
    .write.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'CURATED.DIM_TAXI_ZONES')
    .mode('overwrite')
    .save()
)

(
    df_cat_vendor
    .write.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'CURATED.DIM_VENDOR')
    .mode('overwrite')
    .save()
)

(
    df_cat_rate
    .write.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'CURATED.DIM_RATE_CODE')
    .mode('overwrite')
    .save()
)

(
    df_cat_payment
    .write.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'CURATED.DIM_PAYMENT_TYPE')
    .mode('overwrite')
    .save()
)

(
    df_cat_trip
    .write.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'CURATED.DIM_TRIP_TYPE')
    .mode('overwrite')
    .save()
)

(
    df_cat_saf
    .write.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'CURATED.DIM_STORE_AND_FWD')
    .mode('overwrite')
    .save()
)

(
    df_trips_enriched
    .write.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'CURATED.FCT_TRIPS_ENRICHED')
    .mode('overwrite')
    .save()
)

In [ ]:
# Validaciones rapidas
df_trips_enriched.groupBy('taxi_type').count().orderBy('taxi_type').show()

df_trips_enriched.select(
    'trip_nk', 'taxi_type',
    'pickup_datetime_utc', 'dropoff_datetime_utc',
    'PULocationID', 'pu_borough', 'pu_zone',
    'DOLocationID', 'do_borough', 'do_zone',
    'vendor_id', 'vendor_desc',
    'rate_code_id', 'rate_code_desc',
    'payment_type_code', 'payment_type_desc',
    'trip_type_code', 'trip_type_desc',
    'store_and_fwd_flag_norm', 'store_and_fwd_desc',
    'total_amount', 'trip_duration_minutes'
).show(20, truncate=False)